# Two-lens system with ray-transfer matrices

*optics · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/optics/lens_system)

Paraxial optics is linear algebra: free space and thin lenses are $2\times 2$ matrices acting on (height, angle) ray vectors. The whole system is one chained matrix product applied to a batch of rays, recomputed as the second lens slides away from the first.

**Vector form**

$$M = L_2\,P(d)\,L_1\,P(s),\quad P(d) = \begin{pmatrix}1 & d\\ 0 & 1\end{pmatrix},\ L(f) = \begin{pmatrix}1 & 0\\ -1/f & 1\end{pmatrix},\quad \mathbf y' = M\,\mathbf y$$

**Index (tensor) form**

$$y'_{ri} = M_{ij}\,y_{rj},\qquad M_{im} = L^{(2)}_{ij}\,P^{(d)}_{jk}\,L^{(1)}_{kl}\,P^{(s)}_{lm}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $y_{rj}$ | `rays` | (R, 2) | ray $r$: $j = 0$ height $h$, $j = 1$ angle $\theta$ |
| $P_{ij}(d)$ | `free_space(d)` | (2, 2) | propagation over distance $d$ (batched $d$ gives (..., 2, 2)) |
| $L_{ij}(f)$ | `thin_lens(f)` | (2, 2) | thin lens of focal length $f$ |
| $M_{ik} = E_{ij}M_{jk}$ | `einsum('ij,jk->ik', e, m)` | (2, 2) | chain one more element onto the system (repeated $j$ summed); same as torch.linalg.multi_dot |
| $M_{ij}(z)$ | `einsum('zij,zjk->zik', P, B)` | (Z, 2, 2) | transfer matrix from the object to every axial sample $z$ |
| $y'_{zri} = M_{ij}(z)\,y_{rj}$ | `einsum('zij,rj->zri', m, rays)` | (Z, R, 2) | every ray at every $z$ in one contraction |
| $z_{\text{img}} = -B/D$ | `image_distance(m)` | () | where the height stops depending on the launch angle |

Order matters: the element the light meets first is the rightmost factor. Every factor has determinant 1, so $\det M = 1$ for any arrangement. Watch the image run off to infinity when the intermediate image of lens 1 sits exactly at the focal point of lens 2 ($d = sf_1/(s - f_1) + f_2$), then come back from the other side as a virtual image.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/optics/lens_system/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    s, f1, f2 = params["s"], params["f1"], params["f2"]
    rays = tensors.object_rays(params["h0"], params["half_angle"], params["n_rays"])
    gaps = tensors.separations(params["d_start"], params["d_end"])
    z = tensors.axis(s + params["d_end"] + 60.0)

    heights, image_z, image_h = solve.solve(rays, s, f1, f2, gaps, z)

    m = equation.system_matrix(s, f1, gaps[0].item(), f2)
    print(f"system matrix at d = {gaps[0]:.1f} cm:\n{m}\ndet M = {m.det():.6f}")
    return plot.animate(z, heights, image_z, image_h, gaps, s, params["h0"], out_dir / "lens_system.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/optics/lens_system/equation.py
r"""Physics → tensor translation.

In the paraxial (small-angle) limit a ray at axial position z is a 2-vector y = (height h, angle θ),
and every optical element acts on it linearly: a 2×2 ray-transfer ("ABCD") matrix,

    free space of length d:   P(d) = [[1, d], [0, 1]]       (h grows by θ d, θ unchanged)
    thin lens of focal f:     L(f) = [[1, 0], [-1/f, 1]]    (h unchanged, θ kinked by -h/f)

Light meeting P(s), then L_1, then P(d), then L_2 sees the matrix product (rightmost acts first)

    M = L_2\, P(d)\, L_1\, P(s),    in index form  M_{im} = L^{(2)}_{ij} P^{(d)}_{jk} L^{(1)}_{kl} P^{(s)}_{lm}

and every ray r of a batch is mapped by the same matrix:

    y'_{ri} = M_{ij}\, y_{rj}        (repeated j ⇒ summed: einsum('ij,rj->ri'))

After the last lens the ray height is h' = (A + zC) h + (B + zD) θ. An image forms where h' no longer
depends on the starting angle θ, i.e. at z = -B/D. Since det P = det L = 1, also det M = 1.
"""

import torch


def free_space(d: torch.Tensor) -> torch.Tensor:
    """P(d) = [[1, d], [0, 1]].   d: any shape (...) → (..., 2, 2)"""
    d = torch.as_tensor(d, dtype=torch.float32)
    one, zero = torch.ones_like(d), torch.zeros_like(d)
    return torch.stack([torch.stack([one, d], -1), torch.stack([zero, one], -1)], -2)


def thin_lens(f: float) -> torch.Tensor:
    """L(f) = [[1, 0], [-1/f, 1]]: shape (2, 2)."""
    return torch.tensor([[1.0, 0.0], [-1.0 / f, 1.0]])


def chain(*elements: torch.Tensor) -> torch.Tensor:
    r"""Product of the elements in the order light meets them: M = E_n … E_2 E_1.

    Each step is the matrix product  M_{ik} ← E_{ij} M_{jk}  (repeated j summed).
    Same result as torch.linalg.multi_dot(list(reversed(elements))).
    """
    m = torch.eye(2)
    for e in elements:
        m = torch.einsum("ij,jk->ik", e, m)  # (2, 2)
    return m


def system_matrix(s: float, f1: float, d: float, f2: float) -> torch.Tensor:
    """Object → lens 1 → gap d → lens 2, as one (2, 2) matrix  M = L_2 P(d) L_1 P(s)."""
    return torch.linalg.multi_dot([thin_lens(f2), free_space(d), thin_lens(f1), free_space(s)])


def apply(m: torch.Tensor, rays: torch.Tensor) -> torch.Tensor:
    r"""y'_{ri} = M_{ij} y_{rj}.   m: (2, 2), rays: (R, 2) → (R, 2)"""
    return torch.einsum("ij,rj->ri", m, rays)


def image_distance(m: torch.Tensor) -> torch.Tensor:
    """Distance behind the last element where all rays from one object point meet: z = -B/D."""
    return -m[0, 1] / m[1, 1]


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/optics/lens_system/tensors.py
"""Build the concrete tensors: the fan of rays from the object tip, the axis samples and the lens gaps."""

import torch


def object_rays(height: float, half_angle: float, n_rays: int) -> torch.Tensor:
    """Rays leaving the tip of the object (height h0) at angles in [-half_angle, half_angle].

    rays[r, :] = (h0, θ_r)          ray vector y_{rj}, j ∈ {height, angle}     (R, 2)
    """
    angles = torch.linspace(-half_angle, half_angle, n_rays)  # (R,)
    return torch.stack([torch.full_like(angles, height), angles], dim=-1)


def axis(z_end: float, n_points: int = 300) -> torch.Tensor:
    """Positions along the optical axis where ray heights are evaluated: (Z,)."""
    return torch.linspace(0.0, z_end, n_points)


def separations(d_start: float, d_end: float, n_frames: int = 120) -> torch.Tensor:
    """Lens gap for every animation frame: (F,)."""
    return torch.linspace(d_start, d_end, n_frames)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/optics/lens_system/solve.py
"""Trace the ray fan through the two-lens system for every lens gap, and locate the image."""

import torch

from .equation import chain, free_space, image_distance, system_matrix, thin_lens


def ray_heights(z: torch.Tensor, rays: torch.Tensor, s: float, f1: float, d: float, f2: float) -> torch.Tensor:
    """Height of every ray at every axial position: (Z,), (R, 2) → (Z, R).

    For a point z inside segment k the transfer matrix is  M(z) = P(z - z_k) · B_k,  where B_k is
    everything the light has already passed: B_0 = 1, B_1 = L_1 P(s), B_2 = L_2 P(d) L_1 P(s).
    """
    l1, l2 = thin_lens(f1), thin_lens(f2)
    before = torch.stack([torch.eye(2), chain(free_space(s), l1), chain(free_space(s), l1, free_space(d), l2)])
    starts = torch.tensor([0.0, s, s + d])  # z_k where each segment begins   (3,)

    segment = (z >= s).long() + (z >= s + d).long()  # (Z,) which segment each z lies in
    m = torch.einsum("zij,zjk->zik", free_space(z - starts[segment]), before[segment])  # (Z, 2, 2)  M_ij(z)
    return torch.einsum("zij,rj->zri", m, rays)[..., 0]  # (Z, R)  keep the height component


def solve(
    rays: torch.Tensor, s: float, f1: float, f2: float, gaps: torch.Tensor, z: torch.Tensor
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Return (heights (F, Z, R), image position z_img (F,), image height (F,)) for every gap d."""
    h0 = rays[0, 0]
    heights, image_z, image_h = [], [], []
    for d in gaps.tolist():
        heights.append(ray_heights(z, rays, s, f1, d, f2))
        m = system_matrix(s, f1, d, f2)  # (2, 2)
        z_img = image_distance(m)  # measured from lens 2; negative ⇒ virtual image
        image_z.append(s + d + z_img)
        image_h.append((m[0, 0] + z_img * m[1, 0]) * h0)  # h' = (A + zC) h0
    return torch.stack(heights), torch.stack(image_z), torch.stack(image_h)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/optics/lens_system/plot.py
"""Animate the ray fan while lens 2 slides away from lens 1; mark where the image forms."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif


def animate(
    z: torch.Tensor,
    heights: torch.Tensor,
    image_z: torch.Tensor,
    image_h: torch.Tensor,
    gaps: torch.Tensor,
    s: float,
    h0: float,
    path: Path,
) -> Path:
    zz = z.numpy()  # (Z,)
    h = heights.numpy()  # (F, Z, R)
    zi, hi, d = image_z.numpy(), image_h.numpy(), gaps.numpy()  # (F,) each
    y_max = 4 * h0

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.set_xlim(zz[0], zz[-1])
    ax.set_ylim(-y_max, y_max)
    ax.set_xlabel("z (cm)")
    ax.set_ylabel("height (cm)")
    ax.axhline(0, color="#6b7280", lw=0.8)
    ax.annotate("", (0, h0), (0, 0), arrowprops={"arrowstyle": "-|>", "color": "#111827", "lw": 2})

    colors = plt.get_cmap("viridis")(np.linspace(0, 0.9, h.shape[2]))
    rays = [ax.plot([], [], lw=1.0, color=c)[0] for c in colors]
    lens_kw = {"arrowstyle": "<|-|>", "color": "#2563eb", "lw": 2, "mutation_scale": 12}
    ax.annotate("", (s, 0.9 * y_max), (s, -0.9 * y_max), arrowprops=lens_kw)
    lens2 = ax.annotate("", (s, 0.9 * y_max), (s, -0.9 * y_max), arrowprops=lens_kw)
    (image,) = ax.plot([], [], lw=2.5, color="#dc2626")
    (tip,) = ax.plot([], [], "o", color="#dc2626")
    ax.text(s, -0.98 * y_max, "L1", ha="center", va="bottom", color="#2563eb", fontsize=9)
    name2 = ax.text(s, -0.98 * y_max, "L2", ha="center", va="bottom", color="#2563eb", fontsize=9)

    def update(frame: int):
        for r, line in enumerate(rays):
            line.set_data(zz, h[frame, :, r])
        lens2.xy = (s + d[frame], 0.9 * y_max)
        lens2.xyann = (s + d[frame], -0.9 * y_max)
        name2.set_x(s + d[frame])
        real = zi[frame] > s + d[frame]  # image behind lens 2
        image.set_data([zi[frame], zi[frame]], [0, hi[frame]])
        image.set_linestyle("-" if real else ":")
        tip.set_data([zi[frame]], [hi[frame]])
        kind = "real" if real else "virtual"
        ax.set_title(f"lens gap d = {d[frame]:.1f} cm  →  {kind} image at z = {zi[frame]:.1f} cm")

    ax.set_title("lens gap")  # placeholder so tight_layout leaves room for the title
    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(d), 90), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    's': 20.0,
    'f1': 10.0,
    'f2': 15.0,
    'd_start': 10.0,
    'd_end': 60.0,
    'h0': 1.0,
    'half_angle': 0.08,
    'n_rays': 9
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.optics.lens_system")]:
    del sys.modules[name]
main = importlib.import_module("problems.optics.lens_system.main")
problem_dir = Path("problems/optics/lens_system")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
